# Generate PYQ MCQs in Colab (Gemini, no OpenAI)

1. Get a **free** API key: [Google AI Studio](https://aistudio.google.com/apikey)
2. Upload `corpus.sqlite` from `data/pyq-corpus/` (or zip your ChemCompass `data/pyq-corpus` folder)
3. Set the key in the cell below, run all cells, download `generated_mcqs.json`

In [ ]:
import os, json, sys, subprocess
from pathlib import Path

os.environ['GEMINI_API_KEY'] = ''  # paste key here, or use Colab secrets
# from google.colab import userdata
# os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

!pip install -q httpx fastapi uvicorn python-dotenv

REPO = Path('/content/ChemCompass')  # optional: !git clone your repo
sys.path.insert(0, str(REPO / 'scripts/pyq-llm'))
os.environ['PYQ_CORPUS_DB'] = str(REPO / 'data/pyq-corpus/corpus.sqlite')

assert os.environ['GEMINI_API_KEY'], 'Set GEMINI_API_KEY first'

In [ ]:
import asyncio
from pyq_llm.retrieve import retrieve
from pyq_llm.answer_quality import filter_quality_hits
from pyq_llm.generate_mcq import generate_mcq
from pyq_llm.config import CORPUS_DB

CHAPTER = 'Solutions'
COUNT = 8

async def run():
    out = []
    used = set()
    for i in range(COUNT):
        hits = filter_quality_hits(retrieve(CORPUS_DB, chapter=CHAPTER, query=CHAPTER, limit=25, exclude_ids=used))
        if not hits:
            break
        focus = [hits[i % len(hits)]] + hits[1:4]
        mcq = await generate_mcq(focus, chapter=CHAPTER)
        for sid in mcq.get('sourcePyqIds') or []:
            used.add(sid)
        out.append(mcq)
    return out

mcqs = asyncio.run(run())
Path('generated_mcqs.json').write_text(json.dumps(mcqs, indent=2, ensure_ascii=False), encoding='utf-8')
print('Wrote', len(mcqs), 'MCQs')